# Causal gene regulatory network reconstruction

Cells continuously monitor their internal and external environment and calculate the amount at which each type of protein is needed. This information-processing function is carried out by [gene regulatory networks (GRNs)](https://en.wikipedia.org/wiki/Gene_regulatory_network), which control the rate of production of each protein. Two important classes of regulatory molecules in GRNs are [transcription factors](https://en.wikipedia.org/wiki/Transcription_factor) and [microRNAs](https://en.wikipedia.org/wiki/MicroRNA).

In this tutorial we will use causal inference to reconstruct GRNs from [mRNA](https://en.wikipedia.org/wiki/Messenger_RNA) and [microRNA](https://en.wikipedia.org/wiki/MicroRNA) expression data from the [GEUVADIS study](https://doi.org/10.1038/nature12531), using the methods explained in the [blessing of dimensionality notebook](2_blessing_of_dimensionality_python.ipynb).

## Setup the environment

In [ ]:
import numpy as np
import pandas as pd
import pyarrow as pa
import os

# Setup JuliaCall for calling the Julia BioFindr package.
# Julia must be installed with BioFindr and its dependencies.
# Install JuliaCall with: pip install juliacall
from juliacall import Main as jl

print("JuliaCall setup complete")

In [ ]:
# Load only the Julia packages needed for BioFindr calls.
# DataFrames is needed to pass pandas data to BioFindr; Graphs is used internally by dagfindr!.
jl.seval("""
using BioFindr
using Arrow
using DataFrames
using Graphs
""")
print("Julia packages loaded")

## Load data

This tutorial uses [preprocessed data files](https://github.com/lingfeiwang/findr-data-geuvadis) from the [GEUVADIS study](https://doi.org/10.1038/nature12531). We have mRNA (`t` for transcripts) and microRNA (`m`) expression data:

In [ ]:
# Determine path to the data directory.
# The data should be stored at: <project_root>/data/processed/findr-data-geuvadis/
notebook_dir = os.path.abspath('.')
if os.path.basename(notebook_dir) == 'notebooks':
    project_root = os.path.dirname(notebook_dir)
else:
    project_root = notebook_dir
data_dir = os.path.join(project_root, 'data', 'processed', 'findr-data-geuvadis')

def read_arrow(path):
    """Read an Arrow IPC file into a pandas DataFrame."""
    try:
        return pa.ipc.open_file(path).read_all().to_pandas()
    except pa.lib.ArrowInvalid:
        return pa.ipc.open_stream(path).read_all().to_pandas()

dt = read_arrow(os.path.join(data_dir, 'dt.arrow'))
dm = read_arrow(os.path.join(data_dir, 'dm.arrow'))

print(f"mRNA expression data (dt): {dt.shape[0]} samples x {dt.shape[1]} genes")
print(f"microRNA expression data (dm): {dm.shape[0]} samples x {dm.shape[1]} microRNAs")

We also have genotype data for the strongest [eQTLs](https://en.wikipedia.org/wiki/Expression_quantitative_trait_loci) for a subset of mRNAs and microRNAs:

In [ ]:
dgt = read_arrow(os.path.join(data_dir, 'dgt.arrow'))
dgm = read_arrow(os.path.join(data_dir, 'dgm.arrow'))

print(f"mRNA eQTL genotypes (dgt): {dgt.shape[0]} samples x {dgt.shape[1]} SNPs")
print(f"microRNA eQTL genotypes (dgm): {dgm.shape[0]} samples x {dgm.shape[1]} SNPs")

To construct a GRN, we need a list of transcription factors. We will use a [list](https://humantfs.ccbr.utoronto.ca/allTFs.php) from the [Human Transcription Factors database](https://humantfs.ccbr.utoronto.ca/index.php):

In [ ]:
TFs = pd.read_csv(os.path.join(data_dir, 'TF_names_v_1.01.txt'), header=None, names=['GENE_ID'])
print(f"Number of transcription factors: {len(TFs)}")
TFs.head()

The [preprocessed GEUVADIS data](https://github.com/lingfeiwang/findr-data-geuvadis) has been organized such that each column of the genotype data is the strongest eQTL for the corresponding column in the matching expression data. Usually however, eQTL mapping data will be available in the form of a table with variant IDs, gene IDs, and various eQTL association statistics (see the [original GEUVADIS file](https://www.ebi.ac.uk/biostudies/files/E-GEUV-1/E-GEUV-1/analysis_results/EUR373.gene.cis.FDR5.all.rs137.txt.gz) for an example). Let's artificially create such tables for our data (`p` for "pairs"):

In [ ]:
dpt = pd.DataFrame({
    'SNP_ID':  dgt.columns.tolist(),
    'GENE_ID': dt.columns[:dgt.shape[1]].tolist()
})
dpm = pd.DataFrame({
    'SNP_ID':  dgm.columns.tolist(),
    'GENE_ID': dm.columns[:dgm.shape[1]].tolist()
})
print(f"mRNA eQTL pairs (dpt): {len(dpt)}")
print(f"microRNA eQTL pairs (dpm): {len(dpm)}")
dpt.head()

It is a common observation that TFs don't show strong variation at mRNA level, because their activity is mostly regulated at protein level. As a result, the fraction of TFs with an eQTL is only a bit more than 10%:

In [ ]:
tf_with_eqtl = set(TFs.GENE_ID).intersection(set(dpt.GENE_ID))
print(f"[{len(TFs)} {len(tf_with_eqtl)}]")

Because this tutorial is only for illustration purposes, we stick with these TFs as regulators in our network. A biological way to expand the list is to include upstream signalling molecules as "proxies" for TFs, as they often show more variation at mRNA level (see [this paper](https://www.nature.com/articles/ng1165#Sec7) for an example). Or one may simply accept that all networks inferred from omics data are "functional networks" where individual edges may correspond to indirect physical interactions, and use all genes with eQTLs as potential regulators.

## GRN reconstruction with BioFindr

We start by computing the probabilities of causal interactions from the selected TFs to all other genes using the methods explained in the [blessing of dimensionality notebook](2_blessing_of_dimensionality_python.ipynb). Naturally, we don't have to call the same low-level functions used there. Instead we can call a high-level `findr` function that does everything under the hood. You can read more about this function in the [BioFindr documentation](https://lab.michoel.info/BioFindr.jl/stable/testLLR/) or in the [BioFindr tutorials](https://lab.michoel.info/BioFindrTutorials/). The three first arguments in the call below point to the data and are mandatory (for causal inference). The other arguments are optional: `namesX` tells the function to only use a limited set of regulators (the TFs), and `FDR` sets the false discovery rate threshold for reporting interactions.

Before calling BioFindr's `findr` function we convert the relevant pandas DataFrames to Julia DataFrames. All other data operations (loading, filtering, grouping) remain in Python.

In [ ]:
# Convert pandas DataFrames to Julia DataFrames using the Arrow bridge.
# pa.Table.from_pandas preserves column names and types; Arrow.Table reads them into Julia.
jl.dt_jl  = jl.DataFrame(jl.Arrow.Table(pa.Table.from_pandas(dt)))
jl.dgt_jl = jl.DataFrame(jl.Arrow.Table(pa.Table.from_pandas(dgt)))
jl.dm_jl  = jl.DataFrame(jl.Arrow.Table(pa.Table.from_pandas(dm)))
jl.dgm_jl = jl.DataFrame(jl.Arrow.Table(pa.Table.from_pandas(dgm)))
jl.dpt_jl = jl.DataFrame(jl.Arrow.Table(pa.Table.from_pandas(dpt)))
jl.dpm_jl = jl.DataFrame(jl.Arrow.Table(pa.Table.from_pandas(dpm)))
jl._TF_names = TFs['GENE_ID'].tolist()
jl.seval("TF_names_jl = convert(Vector{String}, _TF_names)")
print("Julia DataFrames ready for BioFindr")

In [ ]:
# Run findr: TF -> mRNA causal interactions
jl.seval("dP_TF_mRNA_jl = findr(dt_jl, dgt_jl, dpt_jl; namesX=TF_names_jl, FDR=0.2)")

dP_TF_mRNA = pd.DataFrame({
    'Source':      list(jl.seval("dP_TF_mRNA_jl.Source")),
    'Target':      list(jl.seval("dP_TF_mRNA_jl.Target")),
    'Probability': np.array(jl.seval("dP_TF_mRNA_jl.Probability")),
    'qvalue':      np.array(jl.seval("dP_TF_mRNA_jl.qvalue"))
})
print(f"TF -> mRNA interactions at FDR=20%: {len(dP_TF_mRNA)}")

In [ ]:
dP_TF_mRNA.head()

We observe a total of more than 8,000 interactions at the FDR=20% threshold. To see how they are distributed over the individual TFs, we do a bit of [data wrangling](https://pandas.pydata.org/docs/user_guide/groupby.html). We see that three TFs are responsible for more than half of the interactions:

In [ ]:
cdf = dP_TF_mRNA.groupby('Source').size().reset_index(name='nrow').sort_values('nrow', ascending=False).reset_index(drop=True)
cdf.head()

<div style="background-color:LightYellow; color:black">
<h3>Exercise</h3> 
     Replace the command below to identify causal microRNA -> microRNA interactions and repeat the steps above to find the number of interactions per microRNA.
</div>

In [ ]:
jl.seval("dP_miRNA_miRNA_jl = findr(dm_jl, dgm_jl, dpm_jl; FDR=0.5)")

dP_miRNA_miRNA = pd.DataFrame({
    'Source':      list(jl.seval("dP_miRNA_miRNA_jl.Source")),
    'Target':      list(jl.seval("dP_miRNA_miRNA_jl.Target")),
    'Probability': np.array(jl.seval("dP_miRNA_miRNA_jl.Probability")),
    'qvalue':      np.array(jl.seval("dP_miRNA_miRNA_jl.qvalue"))
})
print(f"miRNA -> miRNA interactions at FDR=50%: {len(dP_miRNA_miRNA)}")

So far we have reconstructed causal networks *within* a single dataset (mRNA or microRNA expression data) using eQTLs for a subset of variables as causal instruments. We can also reconstruct *bipartite* networks where the source nodes (regulators) come from one dataset and the target nodes from another dataset, as long as the variables in both datasets are measured in the same set of samples. For instance, to identify causal interactions *from* TFs *to* microRNAs, we run:

In [ ]:
jl.seval("dP_TF_miRNA_jl = findr(dm_jl, dt_jl, dgt_jl, dpt_jl; namesX=TF_names_jl, FDR=0.2)")

dP_TF_miRNA = pd.DataFrame({
    'Source':      list(jl.seval("dP_TF_miRNA_jl.Source")),
    'Target':      list(jl.seval("dP_TF_miRNA_jl.Target")),
    'Probability': np.array(jl.seval("dP_TF_miRNA_jl.Probability")),
    'qvalue':      np.array(jl.seval("dP_TF_miRNA_jl.qvalue"))
})
print(f"TF -> miRNA interactions at FDR=20%: {len(dP_TF_miRNA)}")

<div style="background-color:LightYellow; color:black">
<h3>Exercise</h3> 
     Repeat the steps above to find the number of TF -> miRNA interactions per TF. 
</div>

<div style="background-color:LightYellow; color:black">
<h3>Exercise</h3> 
     Replace the command below to identify causal microRNA -> mRNA interactions and repeat the steps above to find the number of interactions per microRNA.
</div>

In [ ]:
dP_miRNA_mRNA = pd.DataFrame({'Source': [], 'Target': [], 'Probability': [], 'qvalue': []})

We can now collect all results in a final dataframe containing our predicted GRN interactions at FDR=20%, sorted by descending probability:

In [ ]:
dP = pd.concat([dP_TF_mRNA, dP_miRNA_miRNA, dP_TF_miRNA, dP_miRNA_mRNA], ignore_index=True)
dP = dP.sort_values('Probability', ascending=False).reset_index(drop=True)
print(f"Total GRN interactions: {len(dP)}")
dP.head()

## Directed acyclic graph reconstruction

In some applications we need graphs that don't contain cycles, so-called [directed acyclic graphs (DAGs)](https://en.wikipedia.org/wiki/Directed_acyclic_graph). For instance, if we want to use the predicted GRN as a structure prior for a [Bayesian network](https://en.wikipedia.org/wiki/Bayesian_network), as done in [this paper](https://doi.org/10.3389/fgene.2019.01196), or if we are looking for a [hierarchical GRN representation](https://doi.org/10.1038/nrg2499). [BioFindr](https://github.com/tmichoel/BioFindr.jl) implements a greedy heuristic to create a DAG from a dataframe of edges, which inserts edges one-by-one into the DAG by descending probability, skipping edges that would introduce a cycle (see the [documentation](https://lab.michoel.info/BioFindr.jl/stable/network/#BioFindr.dagfindr!) for details).

In [ ]:
# Pass the combined edge list from Python to Julia for DAG reconstruction.
# dagfindr! requires a Julia DataFrame and modifies it in place.
jl.dP_jl = jl.DataFrame(jl.Arrow.Table(pa.Table.from_pandas(
    dP[['Source', 'Target', 'Probability', 'qvalue']]
)))
jl.seval("G_all = dagfindr!(dP_jl)")

print(f"Total edges: {len(dP)}")
print(f"Edges in DAG: {int(jl.seval('sum(dP_jl.inDAG_greedy_edges'))}")

The DAG information has been added as a boolean column `inDAG_greedy_edges` indicating whether an edge is included in the DAG or not. Two additional columns, `Source_idx` and `Target_idx`, give a mapping from node names (Gene IDs) to numerical IDs; these IDs are used to identify nodes in `G_all`, a [directed graph object](https://juliagraphs.org/Graphs.jl/dev/core_functions/simplegraphs/#Graphs.SimpleGraphs.SimpleDiGraph) from the [Graphs package](https://juliagraphs.org/Graphs.jl/dev/):

In [ ]:
dP['inDAG_greedy_edges'] = np.array(jl.seval("dP_jl.inDAG_greedy_edges"), dtype=bool)
dP['Source_idx'] = np.array(jl.seval("dP_jl.Source_idx"), dtype=int)
dP['Target_idx'] = np.array(jl.seval("dP_jl.Target_idx"), dtype=int)
dP.head()

<div style="background-color:LightYellow; color:black">
<h3>Exercise</h3> 
     Create a view of dP showing only the interactions <i>not</i> included in the DAG. Do you observe a pattern?
</div>

The regulators (`Source` variables) in our GRN are only a subset of the total set of variables (mRNAs and microRNAs). Since cycles can only occur when the `Target` variable of an edge is also a regulator, it makes more sense to reconstruct DAGs from regulator -> regulator interactions alone. Let's create a new dataframe of these edges:

In [ ]:
regulators = set(dP['Source'].unique())
dP_reg = dP[dP['Target'].isin(regulators)][['Source', 'Target', 'Probability', 'qvalue']].copy().reset_index(drop=True)
print(f"Regulator -> regulator edges: {len(dP_reg)}")
dP_reg.head()

and now call `dagfindr!` again:

In [ ]:
jl.dP_reg_jl = jl.DataFrame(jl.Arrow.Table(pa.Table.from_pandas(
    dP_reg[['Source', 'Target', 'Probability', 'qvalue']]
)))
jl.seval("G_reg = dagfindr!(dP_reg_jl)")

dP_reg['inDAG_greedy_edges'] = np.array(jl.seval("dP_reg_jl.inDAG_greedy_edges"), dtype=bool)
dP_reg['Source_idx'] = np.array(jl.seval("dP_reg_jl.Source_idx"), dtype=int)
dP_reg['Target_idx'] = np.array(jl.seval("dP_reg_jl.Target_idx"), dtype=int)
dP_reg.head()

Let's keep only the DAG edges in `dP_reg` and write both `dP` and `dP_reg` to a file for downstream analyses. **Note:** first create the `data/results/findr-data-geuvadis` directory if it does not exist yet!

In [ ]:
results_dir = os.path.join(project_root, 'data', 'results', 'findr-data-geuvadis')
os.makedirs(results_dir, exist_ok=True)

dP_reg_dag = dP_reg[dP_reg['inDAG_greedy_edges']].reset_index(drop=True)

dP_reg_dag.to_csv(os.path.join(results_dir, 'dP_reg.csv'), index=False)
dP.to_csv(os.path.join(results_dir, 'dP.csv'), index=False)

print(f"Written dP.csv ({len(dP)} rows) to {results_dir}")
print(f"Written dP_reg.csv ({len(dP_reg_dag)} DAG edges) to {results_dir}")

In [ ]:
dP_reg_dag.head()